# Chapter 5b — Time Series
**MADT6004 · Brew Lab BKK case**

A time series has two big components:
- **Trend** — the long-run direction
- **Seasonality** — repeating patterns (weekly, monthly)

Three baseline forecasts every analyst should know:
1. **Naive** — tomorrow = today
2. **Seasonal naive** — next Monday = last Monday
3. **Holt-Winters** — additive trend + seasonal smoothing

Always compare your fancy model against the baselines.


## 0. Bootstrap (Colab + local)

In [ ]:
# Bootstrap — make sure brewlab.db is available, both locally and in Colab.
import os
DB_CANDIDATES = [
    "../../Integrated Data Analytics Exercise/data/brewlab.db",
    "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db",
]
DB_PATH = next((p for p in DB_CANDIDATES if os.path.exists(p)), None)
if DB_PATH is None:
    if not os.path.exists("MADT6004"):
        os.system("git clone -q https://github.com/thanachart/MADT6004.git")
    os.system("pip install -q -r 'MADT6004/Integrated Data Analytics Exercise/requirements.txt'")
    DB_PATH = "MADT6004/Integrated Data Analytics Exercise/data/brewlab.db"
print("DB:", DB_PATH)


## 1. Setup

In [ ]:
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
from statsmodels.tsa.holtwinters import ExponentialSmoothing

conn = sqlite3.connect(DB_PATH)
print("Tables:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])


## 2. Daily revenue at one branch
Pick the *Asoke* branch as the example.

In [ ]:
ts = pd.read_sql("""
SELECT date(t.datetime) AS d, SUM(t.total) AS revenue
FROM transactions t JOIN branches b ON t.branch_id = b.branch_id
WHERE b.name = 'Asoke'
GROUP BY date(t.datetime)
ORDER BY d
""", conn)
ts["d"] = pd.to_datetime(ts["d"])
ts = ts.set_index("d").asfreq("D").fillna(method="ffill")
print(ts.head()); print("Range:", ts.index.min(), "→", ts.index.max())

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts.index, ts["revenue"], color="#0891B2", lw=0.8)
ax.set_title("Asoke — daily revenue")
plt.tight_layout(); plt.show()


## 3. Weekly seasonality check
Average revenue by day of week reveals the weekly pattern.

In [ ]:
dow = ts["revenue"].groupby(ts.index.dayofweek).mean()
labels = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, dow.values, color="#0891B2")
ax.set_title("Mean daily revenue by day of week — Asoke")
ax.set_ylabel("Revenue")
plt.tight_layout(); plt.show()


## 4. Train / test split + three baselines
Hold out the last 30 days. Forecast with naive, seasonal naive, and Holt-Winters.

In [ ]:
train = ts["revenue"].iloc[:-30]
test  = ts["revenue"].iloc[-30:]

# Naive: last training value, repeated
f_naive = pd.Series([train.iloc[-1]] * len(test), index=test.index)

# Seasonal naive: same day of week from prior week
f_snaive = pd.Series(index=test.index, dtype=float)
for i, dt in enumerate(test.index):
    f_snaive.iloc[i] = train.iloc[-(7 - (i % 7))] if i < 7 else f_snaive.iloc[i - 7]

# Holt-Winters with weekly seasonality
hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7).fit()
f_hw = hw.forecast(len(test))

def rmse(y, yhat): return float(np.sqrt(((y - yhat) ** 2).mean()))

print(f"RMSE — naive          : {rmse(test, f_naive):.0f}")
print(f"RMSE — seasonal naive : {rmse(test, f_snaive):.0f}")
print(f"RMSE — Holt-Winters   : {rmse(test, f_hw):.0f}")


## 5. Plot the forecasts

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index[-60:], train.iloc[-60:], color="#374151", lw=0.8, label="train (last 60d)")
ax.plot(test.index, test, color="black", lw=1.2, label="actual")
ax.plot(test.index, f_naive, color="#9CA3AF", lw=1, label="naive")
ax.plot(test.index, f_snaive, color="#D97706", lw=1, label="seasonal naive")
ax.plot(test.index, f_hw, color="#0891B2", lw=1.2, label="Holt-Winters")
ax.legend(); ax.set_title("30-day forecast — Asoke")
plt.tight_layout(); plt.show()


## Discussion prompts
1. Did Holt-Winters beat seasonal naive? By how much?
2. Why is the **naive** baseline often surprisingly hard to beat?
3. How would you extend this forecast to all 12 branches × every SKU? What new problems arise?
